# AI Parking Billing System: IST & Indian Format Validation
This version is fully optimized for **Indian Standard Time (IST)** and strictly validates **Indian License Plates**.

### **Production Features:**
1. **IST Timezone**: Automatically converts Kaggle's UTC time to Indian Standard Time (+5:30).
2. **Regex Validation**: Only accepts plates matching `AA ## AA ####` (e.g., AP22KA2468).
3. **High-Frequency Scanning**: Samples every 5th frame for better accuracy on Exit videos.
4. **Fuzzy Matching**: Matches plates even if minor OCR errors occur.

In [13]:
# 1. Setup
!pip install easyocr opencv-python-headless pandas pytz

import cv2
import easyocr
import pandas as pd
import numpy as np
import re
import os
import pytz
from datetime import datetime

# Initialize OCR
reader = easyocr.Reader(['en'], gpu=True)
print('Robust Environment Ready.')

Robust Environment Ready.


In [14]:
class IndianISTParkingSystem:
    def __init__(self, hourly_rate=2.0):
        self.active_vehicles = {}  # {plate: entry_time}
        self.final_records = []
        self.hourly_rate = hourly_rate
        self.confidence_threshold = 0.3
        self.ist = pytz.timezone('Asia/Kolkata')
        # Regex for Indian Plate: 2 Letters, 2 Digits, 2 Letters, 4 Digits
        self.plate_pattern = re.compile(r'^[A-Z]{2}\d{2}[A-Z]{2}\d{4}$')

    def get_ist_now(self):
        """Helper to get current time in IST."""
        return datetime.now(pytz.utc).astimezone(self.ist)

    def process_video(self, video_path, is_entry=True):
        print(f"\n--- Scanning {'ENTRY' if is_entry else 'EXIT'} Feed: {video_path} ---")
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"Error: Video not found at {video_path}")
            return

        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            frame_count += 1
            if frame_count % 5 == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                h, w = gray.shape
                roi = gray[int(h*0.4):h, 0:w]
                
                results = reader.readtext(roi)
                for (bbox, text, prob) in results:
                    if prob > self.confidence_threshold:
                        clean_plate = ''.join(e for e in text if e.isalnum()).upper()
                        
                        if self.plate_pattern.match(clean_plate):
                            self.handle_event(clean_plate, is_entry)
        
        cap.release()

    def handle_event(self, plate, is_entry):
        now = self.get_ist_now()
        
        # Fuzzy matching for robustness
        match = next((p for p in self.active_vehicles if p in plate or plate in p), None)

        if is_entry:
            if not match:
                self.active_vehicles[plate] = now
                print(f"[IN] Plate: {plate} | IST Entry: {now.strftime('%H:%M:%S')}")
        else:
            if match:
                entry_time = self.active_vehicles.pop(match)
                # In a real-time scenario, exit happens later.
                # For this video-test, we simulate a minimum duration of 1 hour for billing.
                duration_seconds = (now - entry_time).total_seconds()
                duration_hours = max(1.0, duration_seconds / 3600) 
                
                fee = round(duration_hours * self.hourly_rate, 2)
                
                self.final_records.append({
                    'Plate': match,
                    'Entry_IST': entry_time.strftime('%Y-%m-%d %H:%M:%S'),
                    'Exit_IST': now.strftime('%Y-%m-%d %H:%M:%S'),
                    'Fee_INR': f"{fee * 80}" # Converting $2 to roughly ₹160
                })
                print(f"[OUT] Plate: {match} matched! Billing: ₹{fee * 80}")

In [15]:
# --- FINAL EXECUTION WITH YOUR PATHS ---
ENTRY_PATH = '/kaggle/input/datasets/bhanuprasadtelu/vechile-cctv-data/CCTV_Car_Entry_Number_Plate_Generation.mp4'
EXIT_PATH = '/kaggle/input/datasets/bhanuprasadtelu/vechile-cctv-data/Exit_Video_Generation_Complete.mp4'

system = IndianISTParkingSystem(hourly_rate=2.0)

if os.path.exists(ENTRY_PATH):
    system.process_video(ENTRY_PATH, is_entry=True)

if os.path.exists(EXIT_PATH):
    system.process_video(EXIT_PATH, is_entry=False)

if system.final_records:
    df = pd.DataFrame(system.final_records)
    df.to_csv('final_ist_billing.csv', index=False)
    print("\n--- FINAL BILLING REPORT (IST) ---")
    display(df)
else:
    print("\nNo matching plates found between entry and exit videos.")


--- Scanning ENTRY Feed: /kaggle/input/datasets/bhanuprasadtelu/vechile-cctv-data/CCTV_Car_Entry_Number_Plate_Generation.mp4 ---
[IN] Plate: AP22KA2468 | IST Entry: 19:03:52

--- Scanning EXIT Feed: /kaggle/input/datasets/bhanuprasadtelu/vechile-cctv-data/Exit_Video_Generation_Complete.mp4 ---
[OUT] Plate: AP22KA2468 matched! Billing: ₹160.0

--- FINAL BILLING REPORT (IST) ---


,Plate,Entry_IST,Exit_IST,Fee_INR
0,AP22KA2468,2026-04-06 19:03:52,2026-04-06 19:03:58,160.0
